# Monte Carlo vs Q-Learning 비교 실험

이 노트북은 현재 프로젝트의 `environment.py`, `MonteCarlo/mc_agent.py`, `QLearning/q_agent.py`, `compare_training.py` 기준 코드를 노트북 내부에 포함한 제출용 실행 파일입니다.


## 1. 환경 정의

4행 12열 Cliff Walking 환경을 정의합니다. 절벽 또는 미로 밖으로 이동하면 큰 패널티를 받고 episode가 종료되며, 목표에 도착하면 episode가 종료됩니다.


In [ ]:
import numpy as np

# 4 x 12 미로 환경
class Environment():
    
    # 1. 미로밖(절벽), 길, 목적지와 보상 설정
    cliff = -100
    road = -1
    goal = 100
    
    # 2. 목적지 좌표 설정 (우측 하단)
    goal_position = [3,11]
    
    # 2.1 출발 지점 좌표 설정 (좌측 하단)
    start_position = [3,0]
    
    # 3. 보상 리스트 숫자
    # 실제 학습에서 reward 값으로 사용
    reward_list = [[road,road,road,road,road,road,road,road,road,road,road,road],
                   [road,road,road,road,road,road,road,road,road,road,road,road],
                   [road,road,road,road,road,road,road,road,road,road,road,road],
                   [road,cliff,cliff,cliff,cliff,cliff,cliff,cliff,cliff,cliff,cliff,goal]]
    
    # 4. 보상 리스트 문자
    # 상태 종류(길, 절벽, 목표)를 판별할 때 사용
    reward_list1 = [["road","road","road","road","road","road","road","road","road","road","road","road"],
                    ["road","road","road","road","road","road","road","road","road","road","road","road"],
                    ["road","road","road","road","road","road","road","road","road","road","road","road"],
                    ["road","cliff","cliff","cliff","cliff","cliff","cliff","cliff","cliff","cliff","cliff","goal"]]
    
    # 5. 보상 리스트를 array로 설정
    def __init__(self):
        self.reward = np.asarray(self.reward_list)    

    # 6. 선택된 에이전트의 행동 결과 반환
    # 반환값: 다음 위치, 보상, 에피소드 종료 여부
    def move(self, agent, action):
        
        done = False
        
        # 6.1 행동에 따른 좌표 구하기
        new_pos = agent.pos + agent.action[action]

        # 6.2 이동 후 좌표가 미로 밖이면 패널티를 받고 에피소드 종료
        if (new_pos[0] < 0 or new_pos[0] >= self.reward.shape[0] or 
            new_pos[1] < 0 or new_pos[1] >= self.reward.shape[1]):
            observation = agent.set_pos(agent.pos)
            reward = self.cliff
            done = True

        # 6.3 절벽이면 큰 패널티를 받고 에피소드 종료
        elif self.reward_list1[new_pos[0]][new_pos[1]] == "cliff":
            reward = self.cliff
            observation = agent.set_pos(new_pos)
            done = True

        # 6.4 목적지에 도착하면 에피소드 종료
        elif self.reward_list1[new_pos[0]][new_pos[1]] == "goal":
            reward = self.goal
            observation = agent.set_pos(new_pos)
            done = True
            
        # 6.5 이동 후 좌표가 길이라면
        else:
            observation = agent.set_pos(new_pos)
            reward = self.reward[observation[0],observation[1]]
            
        return observation, reward, done


## 2. Monte Carlo 에이전트

에피소드가 끝난 뒤 저장된 경험을 이용해 return G를 계산하고, 방문 횟수 기반 평균으로 Q-table을 업데이트합니다.


In [ ]:
import numpy as np


def e_greedy(q_table, agent, epsilon):
    # epsilon-greedy 방식으로 행동 선택
    pos = agent.get_pos()

    if np.random.rand() <= epsilon:
        return np.random.randint(len(agent.action))

    q_values = q_table[pos[0], pos[1], :]
    max_actions = np.flatnonzero(q_values == np.max(q_values))
    return np.random.choice(max_actions)


class MCAgent:
    action = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])

    """
    몬테카를로 에이전트.
    에피소드를 끝까지 가보고 각 (state, action)에 대해 return G를 incremental average로 계산한다. 
    이후 평균낸 G를 바탕으로 가치함수 테이블을 업데이트한다. 
    """
    # 모델 학습 변수 초기화. 
    def __init__(
        self,
        env,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.9995,
        epsilon_min=0.05,
        first_visit=True,
        **_,
    ):
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.first_visit = first_visit

        # environment.py의 move() 메서드와 호환되는 2장 Agent 구조
        self.pos = np.array(env.start_position)
        self.action_size = len(self.action)
        self.state_shape = env.reward.shape

        self.q_table = np.zeros((env.reward.shape[0], env.reward.shape[1], len(self.action)))
        self.q_visit = np.zeros((env.reward.shape[0], env.reward.shape[1], len(self.action)))
        self.memory = []

    def set_pos(self, position):
        self.pos = np.array(position)
        return self.pos

    def get_pos(self):
        return self.pos

    def select_action(self, state=None):
        """2장/4장 코드의 epsilon-greedy 방식으로 행동 선택"""
        if state is not None:
            self.set_pos(state)
        return e_greedy(self.q_table, self, self.epsilon)

    def append_sample(self, state, action, reward):
        self.memory.append((np.array(state), action, reward))

    def train_model(self):
        """
        2장 Monte Carlo control:
        Q(s,a) <- average(Return(s,a))
        """
        G = 0
        visited = set()

        for state, action, reward in reversed(self.memory):
            G = reward + self.gamma * G
            key = (int(state[0]), int(state[1]), int(action))

            if self.first_visit and key in visited:
                continue

            visited.add(key)
            row, col, act = key
            self.q_visit[row, col, act] += 1
            self.q_table[row, col, act] += (
                (G - self.q_table[row, col, act]) / self.q_visit[row, col, act]
            )

        self.memory = []
        self.decay_epsilon()

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 3. Q-Learning 에이전트

매 step마다 TD target을 이용해 Q-table을 즉시 업데이트합니다.


In [ ]:
import numpy as np


def e_greedy(q_table, agent, epsilon):
    """2장 코드의 epsilon-greedy 방식으로 행동 선택"""
    pos = agent.get_pos()

    if np.random.rand() <= epsilon:
        return np.random.randint(len(agent.action))

    q_values = q_table[pos[0], pos[1], :]
    max_actions = np.flatnonzero(q_values == np.max(q_values))
    return np.random.choice(max_actions)


class QLearningAgent:
    action = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])

    """
    Q-learning 에이전트.

    2장 코드의 TD(0) Q-learning 갱신식을 Cliff Walking 환경에 맞게 옮긴 구현입니다.
    4장의 Q_learning_player처럼 policy와 learn 함수를 분리하되, 상태가 작은
    grid world이므로 신경망 대신 Q-table을 사용합니다.
    """
    def __init__(
        self,
        env,
        learning_rate=0.1,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.01,
        **_,
    ):
        # 외부에서 하이퍼파라미터를 넘겨받을 수 있도록 매개변수화
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        # environment.py의 move() 메서드와 호환되는 2장 Agent 구조
        self.pos = np.array(env.start_position)
        self.action_size = len(self.action)
        self.state_shape = env.reward.shape

        # Cliff Walking 격자 크기에 맞춰 고정 크기의 3차원 NumPy 배열 정의
        self.q_table = np.zeros((env.reward.shape[0], env.reward.shape[1], len(self.action)))

    def set_pos(self, position):
        self.pos = np.array(position)
        return self.pos

    def get_pos(self):
        return self.pos

    def select_action(self, state=None):
        """2장/4장 코드의 epsilon-greedy 방식으로 행동 선택"""
        # 외부에서 state(좌표)를 직접 주입받아 행동 선택 가능
        # np.flatnonzero를 사용하여 최대 Q값을 가진 모든 행동들 중 균등한 확률로 무작위 선택
        if state is not None:
            self.set_pos(state)
        return e_greedy(self.q_table, self, self.epsilon)

    def train_model(self, state, action, reward, next_state, done):
        """
        2장 Q-learning:
        Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]
        """
        row, col = int(state[0]), int(state[1])
        next_row, next_col = int(next_state[0]), int(next_state[1])

        now_q = self.q_table[row, col, action]
        next_q = 0 if done else np.max(self.q_table[next_row, next_col, :])
        target = reward + self.gamma * next_q

        self.q_table[row, col, action] += self.learning_rate * (target - now_q)

    def decay_epsilon(self):
        # 매 에피소드 종료 시점마다 지수 형태로 감쇄하여 epsilon_min 하한선까지 decay
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 4. 학습, 평가, 시각화 함수

두 에이전트를 같은 환경에서 학습시키고, reward 구간 평균과 최종 greedy path 지표를 따로 출력 및 저장합니다.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm


# MAX STEP을 제한하여 학습의 길이를 줄인다. 
MAX_STEPS = 200
# 에피소드 20000개
EPISODES = 20000
EVAL_EPISODES = 50


def get_exploring_start_states(env):
    """2장 Monte Carlo exploring starts를 위한 road 상태 목록"""
    states = []
    for row in range(env.reward.shape[0]):
        for col in range(env.reward.shape[1]):
            if env.reward_list1[row][col] == "road":
                states.append(np.array([row, col]))
    return states


def run_episode(env, agent, algo, train=True, max_steps=MAX_STEPS, start_state=None, first_action=None):
    """하나의 에피소드를 실행하고 보상, 스텝 수, 절벽 추락 횟수, 성공 여부를 반환"""
    state = np.array(env.start_position if start_state is None else start_state)
    agent.set_pos(state)

    # 에피소드별 성능 지표
    total_reward = 0
    steps = 0
    falls = 0
    done = False
    success = False

    while not done and steps < max_steps:
        if steps == 0 and first_action is not None:
            action = first_action
        else:
            action = agent.select_action(state)

        next_state, reward, done = env.move(agent, action)
        next_state = np.array(next_state)
        if done and reward == env.goal:
            success = True

        if train:
            if algo == "MC":
                agent.append_sample(state, action, reward)
            elif algo == "QL":
                agent.train_model(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        steps += 1
        falls += int(reward == env.cliff)

    return total_reward, steps, falls, success


def evaluate_agent(agent, algo, episodes=EVAL_EPISODES):
    """학습이 끝난 에이전트를 greedy 정책으로 평가"""
    env = Environment()
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    # 평가 episode들의 결과를 저장
    rewards = []
    success_steps = []
    falls = 0
    successes = 0

    for _ in range(episodes):
        reward, steps, episode_falls, success = run_episode(env, agent, algo, train=False)
        rewards.append(reward)
        falls += episode_falls

        if success:
            successes += 1
            success_steps.append(steps)

    agent.epsilon = old_epsilon

    return {
        "EvalReward": np.mean(rewards),
        "EvalSteps": np.mean(success_steps) if success_steps else np.nan,
        "EvalFalls": falls,
        "EvalSuccessRate": successes / episodes,
    }


def get_greedy_path(agent, max_steps=MAX_STEPS):
    """학습된 greedy 정책으로 시작점부터 이동한 경로를 기록"""
    env = Environment()
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    state = np.array(env.start_position)
    agent.set_pos(state)

    path = [tuple(state)]
    rewards = []
    actions = []
    done = False
    success = False

    for _ in range(max_steps):
        action = agent.select_action(state)
        next_state, reward, done = env.move(agent, action)
        next_state = np.array(next_state)
        if done and reward == env.goal:
            success = True

        actions.append(action)
        rewards.append(reward)
        path.append(tuple(next_state))

        state = next_state
        if done:
            break

    agent.epsilon = old_epsilon
    return path, actions, rewards, success


def average_range(values, start, end=None):
    """episode 구간의 평균값을 계산"""
    selected = values[start:end]
    if len(selected) == 0:
        return np.nan
    return np.mean(selected)


def format_float(value, width=10):
    """표 출력용 숫자 포맷"""
    if np.isnan(value):
        return f"{'N/A':<{width}}"
    return f"{value:<{width}.2f}"


def summarize_greedy_path(agent):
    """최종 greedy 경로의 보상, 스텝 수, 절벽 추락 횟수, 성공 여부를 정리"""
    env = Environment()
    path, actions, rewards, success = get_greedy_path(agent)

    return {
        "GreedyReward": int(np.sum(rewards)),
        "GreedySteps": len(actions),
        "GreedyFalls": int(np.sum(np.array(rewards) == env.cliff)),
        "GreedySuccess": success,
    }


def train_agent(algo, episodes=EPISODES, seed=0):
    """지정된 알고리즘을 학습하고 episode별 지표를 저장"""
    np.random.seed(seed)
    env = Environment()
    agent = MCAgent(env) if algo == "MC" else QLearningAgent(env)
    exploring_states = get_exploring_start_states(env)

    # 학습 과정 분석을 위한 episode별 기록
    rewards = []
    steps = []
    falls = []
    successes = []

    for _ in tqdm(range(episodes), desc=algo):
        if algo == "MC":
            start_state = exploring_states[np.random.randint(len(exploring_states))]
            first_action = np.random.randint(agent.action_size)
        else:
            start_state = None
            first_action = None

        reward, episode_steps, episode_falls, success = run_episode(
            env,
            agent,
            algo,
            train=True,
            start_state=start_state,
            first_action=first_action,
        )

        if algo == "MC":
            agent.train_model()
        else:
            agent.decay_epsilon()

        rewards.append(reward)
        steps.append(episode_steps)
        falls.append(episode_falls)
        successes.append(int(success))

    return {
        "agent": agent,
        "rewards": np.array(rewards),
        "steps": np.array(steps),
        "falls": np.array(falls),
        "successes": np.array(successes),
        "eval": evaluate_agent(agent, algo),
    }


def summarize_result(algo, result, last_n=50):
    """학습 결과와 평가 결과를 표 출력용 dict로 정리"""
    rewards = result["rewards"]

    return {
        "Algo": algo,
        "TrainRewardMean": np.mean(rewards),
        "TrainRewardLastN": np.mean(rewards[-last_n:]),
        "TrainRewardBest": np.max(rewards),
        "TrainRewardWorst": np.min(rewards),
        "RewardEp1_5000": average_range(rewards, 0, 5000),
        "RewardEp5001_10000": average_range(rewards, 5000, 10000),
        "RewardEp10001_15000": average_range(rewards, 10000, 15000),
        "RewardEp15001_20000": average_range(rewards, 15000, 20000),
        "TrainStepsLastN": np.mean(result["steps"][-last_n:]),
        "TrainFalls": int(np.sum(result["falls"])),
        "TrainSuccessRate": np.mean(result["successes"]),
        **result["eval"],
        **summarize_greedy_path(result["agent"]),
    }


def save_metric_output(filename, text):
    """수치 지표 표를 txt 파일로 저장"""
    os.makedirs("팀플", exist_ok=True)
    path = os.path.join("팀플", filename)
    with open(path, "w", encoding="utf-8") as file:
        file.write(text)
    print(f"\nSaved metric table: {path}")


def print_and_save_metric(title, filename, lines):
    """하나의 지표 표를 콘솔에 출력하고 별도 파일로 저장"""
    text = "\n".join(lines)
    print("\n" + "=" * 92)
    print(title)
    print("=" * 92)
    print(text)
    save_metric_output(filename, text + "\n")


def print_training_reward_metrics(rows):
    """Training Reward 지표를 따로 출력"""
    lines = [
        f"{'Algo':<12} | {'Avg Reward':<11} | {'Last50 Reward':<13} | "
        f"{'Best Reward':<11} | {'Worst Reward':<12}",
        "-" * 70,
    ]

    for row in rows:
        lines.append(
            f"{row['Algo']:<12} | {row['TrainRewardMean']:<11.2f} | "
            f"{row['TrainRewardLastN']:<13.2f} | {row['TrainRewardBest']:<11.2f} | "
            f"{row['TrainRewardWorst']:<12.2f}"
        )

    lines.append("* Last50 Reward is the average reward from the final 50 training episodes.")
    print_and_save_metric("[1. Training Reward]", "training_reward_metrics.txt", lines)


def print_reward_by_phase_metrics(rows):
    """Reward by Phase 지표를 따로 출력"""
    lines = [
        f"{'Algo':<12} | {'Ep 1-5000':<11} | {'Ep 5001-10000':<14} | "
        f"{'Ep 10001-15000':<15} | {'Ep 15001-20000':<15} | {'Last 50':<8}",
        "-" * 86,
    ]

    for row in rows:
        lines.append(
            f"{row['Algo']:<12} | {format_float(row['RewardEp1_5000'], 11)} | "
            f"{format_float(row['RewardEp5001_10000'], 14)} | {format_float(row['RewardEp10001_15000'], 15)} | "
            f"{format_float(row['RewardEp15001_20000'], 15)} | {format_float(row['TrainRewardLastN'], 8)}"
        )

    print_and_save_metric("[2. Reward by Phase]", "reward_by_phase_metrics.txt", lines)


def print_final_greedy_path_metrics(rows):
    """Final Greedy Path 지표를 따로 출력"""
    lines = [
        f"{'Algo':<12} | {'Path Reward':<11} | {'Path Steps':<10} | "
        f"{'Path Falls':<10} | {'Path Success':<12}",
        "-" * 68,
    ]

    for row in rows:
        lines.append(
            f"{row['Algo']:<12} | {row['GreedyReward']:<11} | {row['GreedySteps']:<10} | "
            f"{row['GreedyFalls']:<10} | {str(row['GreedySuccess']):<12}"
        )

    lines.append("* Final Greedy Path is measured with epsilon=0 from the fixed start position.")
    print_and_save_metric("[3. Final Greedy Path]", "final_greedy_path_metrics.txt", lines)


def print_summary(rows):
    """두 알고리즘의 주요 수치 지표를 각각 따로 출력"""
    print_training_reward_metrics(rows)
    print_reward_by_phase_metrics(rows)
    print_final_greedy_path_metrics(rows)


def plot_training(mc_result, ql_result):
    """학습 중 기록된 reward, steps, success, falls를 그래프로 저장"""
    os.makedirs("팀플", exist_ok=True)

    plt.figure(figsize=(14, 8))

    plt.subplot(2, 2, 1)
    plt.plot(mc_result["rewards"], label="Monte Carlo")
    plt.plot(ql_result["rewards"], label="Q-Learning")
    plt.title("Training Reward")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.legend()

    plt.subplot(2, 2, 2)
    plt.plot(mc_result["steps"], label="Monte Carlo")
    plt.plot(ql_result["steps"], label="Q-Learning")
    plt.title("Training Steps")
    plt.xlabel("Episode")
    plt.ylabel("Steps")
    plt.legend()

    plt.subplot(2, 2, 3)
    plt.plot(np.cumsum(mc_result["successes"]) / (np.arange(len(mc_result["successes"])) + 1), label="Monte Carlo")
    plt.plot(np.cumsum(ql_result["successes"]) / (np.arange(len(ql_result["successes"])) + 1), label="Q-Learning")
    plt.title("Cumulative Success Rate")
    plt.xlabel("Episode")
    plt.ylabel("Success Rate")
    plt.legend()

    plt.subplot(2, 2, 4)
    plt.plot(np.cumsum(mc_result["falls"]), label="Monte Carlo")
    plt.plot(np.cumsum(ql_result["falls"]), label="Q-Learning")
    plt.title("Cumulative Cliff Falls")
    plt.xlabel("Episode")
    plt.ylabel("Falls")
    plt.legend()

    plt.tight_layout()
    plt.savefig("팀플/learning_comparison_mc_vs_ql.png")


def plot_policy_path(agent, title, filename):
    """학습된 greedy 정책의 실제 이동 경로를 격자 그림으로 저장"""
    os.makedirs("팀플", exist_ok=True)

    env = Environment()
    path, actions, rewards, success = get_greedy_path(agent)

    rows, cols = env.reward.shape
    color_grid = np.zeros((rows, cols))

    for row in range(rows):
        for col in range(cols):
            if env.reward_list1[row][col] == "cliff":
                color_grid[row, col] = -1
            elif env.reward_list1[row][col] == "goal":
                color_grid[row, col] = 2

    for row, col in path:
        if env.reward_list1[row][col] == "road":
            color_grid[row, col] = 1

    start = tuple(env.start_position)
    goal = tuple(env.goal_position)
    color_grid[start] = 3
    color_grid[goal] = 2

    cmap = plt.matplotlib.colors.ListedColormap([
        "#ef4444",  # cliff
        "#f8fafc",  # road
        "#60a5fa",  # path
        "#22c55e",  # goal
        "#facc15",  # start
    ])
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
    norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

    plt.figure(figsize=(14, 5))
    plt.imshow(color_grid, cmap=cmap, norm=norm)
    plt.xticks(range(cols))
    plt.yticks(range(rows))
    plt.grid(which="major", color="#334155", linewidth=1)
    plt.tick_params(bottom=False, left=False)

    for idx, (row, col) in enumerate(path):
        label = "S" if (row, col) == start else "G" if (row, col) == goal else str(idx)
        plt.text(col, row, label, ha="center", va="center", color="#0f172a", fontsize=10, fontweight="bold")

    action_symbol = {0: "↑", 1: "→", 2: "↓", 3: "←"}
    for (row, col), action in zip(path[:-1], actions):
        plt.text(col + 0.28, row - 0.28, action_symbol[action], ha="center", va="center", color="#111827", fontsize=12)

    total_reward = sum(rewards)
    plt.title(f"{title} Greedy Path | Success: {success} | Steps: {len(actions)} | Reward: {total_reward}")
    plt.tight_layout()
    plt.savefig(filename)
    return path, actions, rewards, success


def plot_all_policy_paths(mc_result, ql_result):
    """MC와 Q-learning의 최종 greedy 경로 그림을 모두 저장"""
    mc_path = plot_policy_path(
        mc_result["agent"],
        "Monte Carlo",
        "팀플/monte_carlo_greedy_path.png",
    )
    ql_path = plot_policy_path(
        ql_result["agent"],
        "Q-Learning",
        "팀플/q_learning_greedy_path.png",
    )
    return mc_path, ql_path


## 5. 실험 실행

전체 20000 episode를 학습한 뒤 수치 지표와 greedy path 그림을 `팀플` 폴더에 저장합니다.


In [ ]:
# 학습 실행 및 결과 출력
mc_result = train_agent("MC", episodes=EPISODES, seed=0)
ql_result = train_agent("QL", episodes=EPISODES, seed=0)

rows = [
    summarize_result("Monte Carlo", mc_result),
    summarize_result("Q-Learning", ql_result),
]

print_summary(rows)
plot_training(mc_result, ql_result)
plot_all_policy_paths(mc_result, ql_result)
print("\n* Plot saved as '팀플/learning_comparison_mc_vs_ql.png'.")
print("* Greedy paths saved as '팀플/monte_carlo_greedy_path.png' and '팀플/q_learning_greedy_path.png'.")
